# CP2 Week 8 -- Timing & Performance

**Course:** Computer Programming 2 (CP2)
**Prerequisites:** Weeks 1-7
**Focus:** timeit, profiling, comparing approaches

## Learning Objectives
- Measure code performance with timeit
- Compare different approaches objectively
- Profile your pipeline to find bottlenecks
- Write performance reports

In [ ]:
# === SETUP (run this first) ===
# If running in Google Colab, uncomment and run the lines below:
# !git clone https://github.com/YOUR-ORG/YOUR-REPO.git
# %cd YOUR-REPO

import sys, os
# Add src to path so we can import project modules
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')
elif os.path.exists('../../src'):
    sys.path.insert(0, '../../src')

print("Setup complete! Ready to work.")

---
## Part 1: Measuring Performance With timeit

"I think this is faster" is not good enough. MEASURE it.

In [ ]:
import timeit

def method_loop(data):
    result = []
    for x in data:
        result.append(x ** 2)
    return result

def method_comprehension(data):
    return [x ** 2 for x in data]

def method_map(data):
    return list(map(lambda x: x ** 2, data))

def method_numpy(data):
    import numpy as np
    return (np.array(data) ** 2).tolist()

data = list(range(10000))

methods = [
    ("Loop", method_loop),
    ("Comprehension", method_comprehension),
    ("Map", method_map),
    ("NumPy", method_numpy),
]

print("Timing 4 approaches (100 runs each):")
results = []
for name, func in methods:
    t = timeit.timeit(lambda: func(data), number=100)
    results.append((name, t))
    print("  " + name.ljust(15) + ": " + str(round(t, 4)) + "s")

fastest = min(results, key=lambda x: x[1])
print("\nFastest: " + fastest[0])

**Expected Output:**
```
Timing 4 approaches (100 runs each):
  Loop           : ~0.25s
  Comprehension  : ~0.18s
  Map            : ~0.22s
  NumPy          : ~0.12s

Fastest: NumPy (approximate)
```

---
## Part 2: Timing Your Pipeline Stages

In [ ]:
import time

def time_function(func, *args, n_runs=5):
    """Time a function and return statistics."""
    times = []
    for _ in range(n_runs):
        start = time.time()
        result = func(*args)
        elapsed = time.time() - start
        times.append(elapsed)
    
    return {
        "mean_ms": round(sum(times) / len(times) * 1000, 2),
        "min_ms": round(min(times) * 1000, 2),
        "max_ms": round(max(times) * 1000, 2),
    }

# Example: time different cleaning approaches
def clean_simple(data):
    return [r for r in data if r.get("value", "") != ""]

def clean_typed(data):
    result = []
    for r in data:
        try:
            val = float(r.get("value", ""))
            if 0 <= val <= 100:
                result.append({**r, "value": val})
        except (ValueError, TypeError):
            pass
    return result

test_data = [{"id": i, "value": str(i * 1.5)} for i in range(10000)]

for name, func in [("Simple", clean_simple), ("Typed", clean_typed)]:
    stats = time_function(func, test_data)
    print(name + ": mean=" + str(stats["mean_ms"]) + "ms")

**Expected Output:**
```
Simple: mean=~2ms
Typed: mean=~8ms
(approximate)
```

---
## Part 3: Profiling Your Pipeline

Profile the entire pipeline to find which stage is the bottleneck.

In [ ]:
import time

def profile_pipeline(data_size=1000):
    """Profile each pipeline stage and print results."""
    # Create test data
    import random
    random.seed(42)
    test_data = [
        {"id": i, "value": str(round(random.gauss(50, 20), 2))}
        for i in range(data_size)
    ]
    
    stages = []
    
    # Stage 1: Load (simulate)
    start = time.time()
    data = list(test_data)  # simulate loading
    stages.append(("load_data", time.time() - start))
    
    # Stage 2: Validate
    start = time.time()
    for row in data[:5]:
        assert "id" in row and "value" in row
    stages.append(("validate", time.time() - start))
    
    # Stage 3: Clean
    start = time.time()
    cleaned = []
    for row in data:
        try:
            v = float(row["value"])
            if 0 <= v <= 100:
                cleaned.append({**row, "value": v})
        except (ValueError, TypeError):
            pass
    stages.append(("clean_data", time.time() - start))
    
    # Stage 4: Analyze
    start = time.time()
    values = [r["value"] for r in cleaned]
    mean_v = sum(values) / len(values) if values else 0
    std_v = (sum((x - mean_v)**2 for x in values) / len(values))**0.5 if values else 0
    stages.append(("analyze", time.time() - start))
    
    # Print profile
    total = sum(t for _, t in stages)
    print("Pipeline Profile (" + str(data_size) + " rows):")
    print("-" * 45)
    for name, t in stages:
        pct = round(t / total * 100, 1) if total > 0 else 0
        ms = round(t * 1000, 2)
        bar = "#" * int(pct / 2)
        print("  " + name.ljust(15) + str(ms).rjust(8) + "ms  "
              + str(pct).rjust(5) + "% " + bar)
    print("-" * 45)
    print("  " + "TOTAL".ljust(15) + str(round(total * 1000, 2)).rjust(8) + "ms")
    
    return stages

profile_pipeline(1000)
print()
profile_pipeline(10000)

**Expected Output:**
```
Pipeline Profile (1000 rows):
---------------------------------------------
  load_data         X.XXms   XX.X% ##
  validate          X.XXms    X.X% 
  clean_data        X.XXms   XX.X% #####
  analyze           X.XXms   XX.X% ##
---------------------------------------------
  TOTAL             X.XXms
(approximate)
```

---
## Part 4: Writing a Performance Report

In [ ]:
def generate_performance_report(stages, data_size):
    """Generate a text performance report."""
    total = sum(t for _, t in stages)
    slowest = max(stages, key=lambda x: x[1])
    
    lines = []
    lines.append("Performance Report")
    lines.append("  Data size: " + str(data_size) + " rows")
    lines.append("  Total time: " + str(round(total * 1000, 2)) + " ms")
    lines.append("  Bottleneck: " + slowest[0] + " ("
                 + str(round(slowest[1] * 1000, 2)) + " ms)")
    lines.append("")
    lines.append("  Recommendations:")
    if slowest[0] == "clean_data":
        lines.append("  - Consider NumPy for numeric operations")
        lines.append("  - Reduce number of cleaning rules if possible")
    elif slowest[0] == "analyze":
        lines.append("  - Use NumPy for statistical calculations")
    lines.append("  - Profile again after optimizations")
    
    return "\n".join(lines)

stages = profile_pipeline(5000)
print()
print(generate_performance_report(stages, 5000))

### Common Mistakes: Timing

**Mistake 1:** Timing code that runs only once -- results vary wildly.

**Fix:** Use timeit with number=100+ or average over multiple runs.

**Mistake 2:** Including print statements in timed code -- I/O is slow.

**Fix:** Remove or silence print statements before timing.

**Mistake 3:** Comparing times on different machines or with different data sizes.

**Fix:** Always compare on the same machine with the same data.


### Key Takeaway

- Always measure, never guess about performance
- timeit is more accurate than time.time() for short operations
- Profile your pipeline to find the slowest stage
- Faster is not always better -- correctness first
- Document performance with timing reports

---
## Homework: 12 Exercises

### Review (1-4)

In [ ]:
# HW1: Time your pipeline load_data function.


In [ ]:
# HW2: Time clean_data with different configs.


In [ ]:
# HW3: Compare list comprehension vs loop for your data.


In [ ]:
# HW4: Which pipeline stage is slowest? Measure each one.


### Practice (5-8)

In [ ]:
# HW5: Create a timing report for all pipeline stages.


In [ ]:
# HW6: Time the same function with different data sizes.


In [ ]:
# HW7: Compare dict.get() vs try/except for missing keys.


In [ ]:
# HW8: Write a decorator that automatically times functions.


### Challenge (9-11)

In [ ]:
# HW9: Create a performance comparison chart (text-based).


In [ ]:
# HW10: Profile memory usage (sys.getsizeof) of different approaches.


In [ ]:
# HW11: Optimize the slowest stage of your pipeline.


### Mini-Project

In [ ]:
# HW12: Write a complete performance report for your pipeline.
# Include: timing per stage, speedup from optimizations, recommendations.


---
## Reflection (Required - Complete before submitting)

In the cell below, write 3-5 bullet points:
1. **What I learned today:**
2. **What was hardest:**
3. **What I still don't understand:**
4. **What I will review before next week:**
5. **If I used AI tools, what for:**

In [ ]:
# YOUR REFLECTION (write as comments or as a multi-line string)
reflection = """
- What I learned:
- What was hardest:
- What I still don't understand:
- What I will review:
- AI tools used (if any):
"""
print(reflection)